In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import find_peaks
import datetime
import multiprocess as mp
import pickle
import time
sys.path.insert(0, "./cellular_cycle") # add path to the model

from cell_cycle_death_model import model
from model_fxns import *
from pysb.simulator import ScipyOdeSimulator

### Get number of cpus available for multiprocessing

In [2]:
# Set number of cores to use for multiprocess
# number of processes to reserve; set to either 90% or n_proc - 1, whichever is greater
if mp.cpu_count() > 1:
    PROCESSES = mp.cpu_count() - 1
if mp.cpu_count() > 10:
    PROCESSES = mp.cpu_count() - int(round(mp.cpu_count()*0.1,0))
 
print(f'Will use {PROCESSES} cores for multiprocessing')

# Where to save large simulation output files
SAVE_DIR = Path(Path.home(),'ECM_mod_output')
    
ODE_DIR = Path(Path.home(),'ECM_mod_output/ODE_output')
ODE_DIR.mkdir(parents=True, exist_ok=True)

SAVEFILES = True # Save simulation output files
OVERWRITE = False # Overwrite existing simulation files
SET_JPARAM = False # Set the best fit parameters from PSO

CURDATE = datetime.datetime.now().strftime('%Y%m%d')

Will use 11 cores for multiprocessing


In [3]:
# Importing the parameters:
new_param = pd.read_csv('model_data/parameters.csv')
for key in new_param.keys():
    try:
        if model.parameters[key].value != new_param[key][0]:
            print(f'Changing {key} from {model.parameters[key].value} to {new_param[key][0]}')
            model.parameters[key].value = new_param[key]
    except:
        print(f'{key} not found in model parameters')
model.parameters['omega'].value = 200

In [4]:
t = np.linspace(0, 1000, 10001)
t_0 = np.linspace(0, 2000, 20001)

model.parameters['AKT'].value = 1
model.parameters['ERK'].value = 1
model.parameters['FAK'].value = 0.6

In [5]:
if SET_JPARAM:
    with open('fit_data/pso/frac_prolif_parameters_pso.pkl', 'rb') as f:
        frac_prolif_parameters_pso = pickle.load(f)
    new_param_pso_sort = frac_prolif_parameters_pso.sort_values([('cost', 'total cost likelihood')])
    new_params = new_param_pso_sort['model parameters']
    # Importing the parameters:
    ii = 0
    for key in new_params.keys():
        try:
            if model.parameters[key].value != new_params[key].iloc[ii]:
                model.parameters[key].value = new_params[key].iloc[ii]
        except:
            print(key)
    model.parameters['omega'].value = 200

In [6]:
initial_concentrations = pd.read_csv('model_data/initial_concentrations.csv')
for key in initial_concentrations.keys():
    try:
        model.parameters[key].value = initial_concentrations[key][0] / model.parameters['omega'].value
    except:
        print(f'{key} not found in model parameters')

In [7]:
integrator_options = {
    "rtol": 1e-12,
    "atol": 1e-12
}

In [8]:
##ERL
treatment = 'ERL'
FAKs = list(np.linspace(0.4,0.6,500))
FAKs.extend(list(np.linspace(0.6,1,500)))

AKTs = list(np.linspace(0.3,0.6,500))
AKTs.extend([0.6]*500)

ERKs = [0.2]*1000

vals=np.linspace(0,999,1000)

conds = [(AKT, ERK, FAK, val) for AKT, ERK, FAK, val in zip(AKTs, ERKs, FAKs, vals)]

In [9]:
def run_odes(AKT, ERK, FAK, val, treatment=treatment, np=np, model=model, ScipyOdeSimulator=ScipyOdeSimulator, find_peaks=find_peaks, Path=Path, 
             integrator_options=integrator_options, SET_JPARAM=SET_JPARAM, ODE_DIR=ODE_DIR, save=SAVEFILES, overwrite=OVERWRITE): #, ii=ii
    model.parameters['AKT'].value = AKT
    model.parameters['ERK'].value = ERK
    model.parameters['FAK'].value = FAK
    t = np.linspace(0, 2000, 20001)    

    sim_ode = ScipyOdeSimulator(model, t, verbose=False, compiler="python", integrator_options=integrator_options).run()
    y_out = sim_ode.observables

    peaks_cdc, _ = find_peaks(y_out['obsCdc20'], prominence = 0.6, distance = 10)
    peaks_half = peaks_cdc[peaks_cdc < len(y_out)/2]
    peaks_half2 = peaks_cdc[peaks_cdc > len(y_out)/2]
    
    if len(peaks_cdc) > 7:
        variable_range=y_out[peaks_cdc[-6]:peaks_cdc[-1]]
    else:
        variable_range = y_out[19000:20000]        
    
    tmp = int(val)
    if SET_JPARAM:
        filename = Path(ODE_DIR,f'{treatment}_psoparam_{ii}_{tmp:03}.csv')
    else:
        filename = Path(ODE_DIR,f'{treatment}_range_{tmp:03}.csv')
    
    if filename.exists() and not overwrite:
        print(f'{filename} exists, skipping')

    if save and not filename.exists() or overwrite:
        print(f'Saving {filename}')
        np.savetxt(filename, variable_range, delimiter=',')

    if len(peaks_half2) > 0:
        IMTs = [t[y] - t[x] for x, y in zip(peaks_half2, peaks_half2[1:])]
        meanIMT = sum(IMTs) / len(IMTs)
    else:
        meanIMT = 0
   
    return meanIMT        

In [10]:
conds = [(AKT, ERK, FAK, val) for AKT, ERK, FAK, val in zip(AKTs, ERKs, FAKs, vals)]

In [11]:
if SAVEFILES or OVERWRITE:
    start = time.time()
    with mp.Pool(processes = PROCESSES) as p:
        results = p.starmap(run_odes, conds)
    end = time.time()
    print(f'Processing took {round((end - start)/60,2)} minutes')

/Users/darren/ECM_mod_output/ODE_output/ERL_range_000.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_023.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_046.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_069.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_092.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_115.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_138.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_161.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_184.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_207.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_230.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_001.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/ERL_range_024.csv exists, skipping
/Users/darren/ECM_mod_out

In [12]:
if SET_JPARAM:
    filename = Path(ODE_DIR,f'{treatment}_pso_param_IMT.csv')
else:
    filename = Path(ODE_DIR,f'{treatment}_IMT.csv')
if filename.exists() and not OVERWRITE:
    print(f'{filename} exists, skipping')
if SAVEFILES and not filename.exists() or OVERWRITE:
    print(f'Saving {filename}')
    np.savetxt(filename, results, delimiter=',')

/Users/darren/ECM_mod_output/ODE_output/ERL_IMT.csv exists, skipping


In [13]:
##DMSO
treatment = 'DMSO'
FAKs = list(np.linspace(0.4,0.6,500))
FAKs.extend(list(np.linspace(0.6,1,500)))

AKTs = list(np.linspace(0.6,1,500))
AKTs.extend([1]*500)

ERKs = [1]*1000

In [14]:
conds = [(AKT, ERK, FAK, val, treatment) for AKT, ERK, FAK, val in zip(AKTs, ERKs, FAKs, vals)]

In [15]:
start = time.time()
with mp.Pool(processes = PROCESSES) as p:
    results = p.starmap(run_odes, conds)
end = time.time()
print(f'Processing took {round((end - start)/60,2)} minutes')

/Users/darren/ECM_mod_output/ODE_output/DMSO_range_023.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/DMSO_range_207.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/DMSO_range_230.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/DMSO_range_000.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/DMSO_range_184.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/DMSO_range_069.csv exists, skipping/Users/darren/ECM_mod_output/ODE_output/DMSO_range_161.csv exists, skipping

/Users/darren/ECM_mod_output/ODE_output/DMSO_range_115.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/DMSO_range_046.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/DMSO_range_138.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/DMSO_range_092.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/DMSO_range_231.csv exists, skipping
/Users/darren/ECM_mod_output/ODE_output/DMSO_range_208.csv exists, skipping
/Users/darre

In [16]:
if SET_JPARAM:
    filename = Path(ODE_DIR,f'{treatment}_pso_param_IMT.csv')
else:
    filename = Path(ODE_DIR,f'{treatment}_IMT.csv')
if filename.exists() and not OVERWRITE:
    print(f'{filename} exists, skipping')
if SAVEFILES and not filename.exists() or OVERWRITE:
    print(f'Saving {filename}')
    np.savetxt(filename, results, delimiter=',')

/Users/darren/ECM_mod_output/ODE_output/DMSO_IMT.csv exists, skipping
